# Session 5 lab — Data warehouses and dimensional modelling

The same trips, read two ways and modelled two ways. In part A one query runs on
PostgreSQL and on DuckDB, and you measure what each one reads. In part B you build
a small star schema from the normalised tables and ask it one of today's requests.

To hand in: the two measurements from part A, `sql/star_schema.sql` with its grain
stated, and two sentences comparing the queries in part B.

In [2]:
import re
from pathlib import Path
from typing import LiteralString

import duckdb
import pandas as pd
import psycopg
from psycopg.sql import SQL, Identifier


def find(*candidates: str) -> Path:
    """This notebook runs from its own folder, or from anywhere in the container."""
    for candidate in candidates:
        if Path(candidate).exists():
            return Path(candidate).resolve()
    raise SystemExit(f"not found: {candidates[0]}")


SQL_DIR = find("sql", "session-05/sql", "/app/labs/session-05/sql")
LABS = SQL_DIR.parents[1]
DATA = find("../../data", "data", "/app/data")
WHOLE = DATA / "nyc-taxi-parquet" / "trips.parquet"
BY_DAY = DATA / "nyc-taxi-parquet" / "trips_by_day"
if not WHOLE.exists():
    raise SystemExit("no Parquet files yet: run prepare_parquet.py, see the README")

PG = psycopg.connect("host=postgres port=5432 user=labs password=labs dbname=labs")
PG.autocommit = True
PG.execute("SET TIME ZONE 'America/New_York'")

# DuckDB has no server: it runs here, inside this notebook's Python process.
DUCK = duckdb.connect()


def sql(statement: LiteralString) -> pd.DataFrame:
    """Run one statement on PostgreSQL, and show whatever rows come back."""
    if not statement.strip():
        print("nothing to run yet — write your query between the quotes")
        return pd.DataFrame()
    with PG.cursor() as cursor:
        cursor.execute(statement)
        if cursor.description is None:
            return pd.DataFrame()
        columns = [column.name for column in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=columns)


def run_file(path: Path) -> None:
    """Run every statement in a .sql file on PostgreSQL."""
    PG.execute(path.read_bytes())


def ensure_database() -> None:
    """Load session 4's database, unless it is still there from last week."""
    found = PG.execute("SELECT to_regclass('trips') IS NOT NULL").fetchone()
    if found and found[0]:
        has_rows = PG.execute("SELECT EXISTS (SELECT 1 FROM trips)").fetchone()
        if has_rows and has_rows[0]:
            print("PostgreSQL: session 4's tables are loaded")
            return

    print("PostgreSQL: loading session 4's tables, about 20 seconds")
    run_file(LABS / "session-03" / "sql" / "schema.sql")
    run_file(LABS / "session-04" / "sql" / "split_fares.sql")
    # Session 4 loads the payments in which some fares were split.
    tables = {
        "zones": "zones",
        "drivers": "drivers",
        "riders": "riders",
        "driver_balances": "driver_balances",
        "trips": "trips",
        "payments": "payments_split",
    }
    for table, file_name in tables.items():
        copy_in = SQL("COPY {} FROM STDIN WITH (FORMAT csv, HEADER)")
        path = DATA / "nyc-taxi-normalized" / f"{file_name}.csv"
        with path.open() as file, PG.cursor() as cursor:
            with cursor.copy(copy_in.format(Identifier(table))) as copy:
                while chunk := file.read(1 << 20):
                    copy.write(chunk)
    PG.execute("ANALYZE")


ensure_database()

PostgreSQL: session 4's tables are loaded


## Part A — One query, two engines (10 min)

The average fare for each hour of the day, over all 1,000,000 trips. PostgreSQL
reads its `trips` table; DuckDB reads the same rows from `trips.parquet`. Both
cells run the query twice and report the second run, so neither is timed reading
from a cold disk.

In [3]:
QUERY = """
SELECT extract(hour FROM pickup_at) AS hour,
       avg(fare)                    AS avg_fare
FROM {trips}
GROUP BY hour
ORDER BY hour
"""


def postgres_plan(query: LiteralString) -> None:
    """EXPLAIN ANALYZE on PostgreSQL, then the time and the bytes it read."""
    PG.execute(f"EXPLAIN ANALYZE {query}")
    for (line,) in PG.execute(f"EXPLAIN (ANALYZE, BUFFERS) {query}").fetchall():
        print(line)
    report = PG.execute(f"EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON) {query}").fetchone()
    assert report is not None
    plan = report[0][0]
    pages = plan["Plan"]["Shared Hit Blocks"] + plan["Plan"]["Shared Read Blocks"]
    print(f"\nPostgreSQL: {plan['Execution Time']:.0f} ms, "
          f"{pages:,} pages of 8 kB = {pages * 8192 / 1e6:.1f} MB read")


postgres_plan(QUERY.format(trips="trips"))

Finalize GroupAggregate  (cost=67769.17..189920.92 rows=689288 width=64) (actual time=141.298..189.637 rows=24 loops=1)
  Group Key: (EXTRACT(hour FROM pickup_at))
  Buffers: shared hit=2551 read=8793, temp read=2657 written=2671
  ->  Gather Merge  (cost=67769.17..173331.60 rows=833334 width=64) (actual time=140.099..189.578 rows=72 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        Buffers: shared hit=2551 read=8793, temp read=2657 written=2671
        ->  Partial GroupAggregate  (cost=66769.15..76144.15 rows=416667 width=64) (actual time=90.692..135.827 rows=24 loops=3)
              Group Key: (EXTRACT(hour FROM pickup_at))
              Buffers: shared hit=2551 read=8793, temp read=2657 written=2671
              ->  Sort  (cost=66769.15..67810.81 rows=416667 width=38) (actual time=89.387..111.208 rows=333333 loops=3)
                    Sort Key: (EXTRACT(hour FROM pickup_at))
                    Sort Method: external merge  Disk: 7728kB
                    B

In [4]:
def duckdb_plan(query: str) -> None:
    """EXPLAIN ANALYZE on DuckDB: the plan reads from the bottom up."""
    DUCK.sql(query).fetchall()
    print(DUCK.sql(f"EXPLAIN ANALYZE {query}").fetchall()[0][1])


duckdb_plan(QUERY.format(trips=f"'{WHOLE}'"))

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE  SELECT extract(hour FROM pickup_at) AS hour,        avg(fare)                    AS avg_fare FROM '/app/data/nyc-taxi-parquet/trips.parquet' GROUP BY hour ORDER BY hour 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0077s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        

DuckDB's plan does not print bytes. It prints which columns it read: find
`Projections` in the `TABLE_SCAN` box at the bottom, and list those columns
below. The cell adds up what they occupy in the file.

In [5]:
COLUMNS_READ = []  # e.g. ["some_column", "another_column"]

sizes = DUCK.sql(f"""
    SELECT path_in_schema AS column_name, sum(total_compressed_size) AS bytes
    FROM parquet_metadata('{WHOLE}')
    GROUP BY column_name
""").df().set_index("column_name")["bytes"]

read = sizes[sizes.index.isin(COLUMNS_READ)].sum()
print(f"DuckDB: {read / 1e6:.1f} MB read, of {sizes.sum() / 1e6:.1f} MB in the file")
print(f"in the file, by column (MB):\n{(sizes / 1e6).round(1).to_string()}")

DuckDB: 0.0 MB read, of 24.9 MB in the file
in the file, by column (MB):
column_name
fare               1.6
distance_km        1.6
driver_id          1.3
pickup_at          5.0
dropoff_at         5.4
tip                1.5
rider_id           2.5
pickup_zone_id     1.0
trip_id            4.0
dropoff_zone_id    1.0


Fill in, from the output above:

| | Time (ms) | Data read (MB) |
|---|---|---|
| PostgreSQL | | |
| DuckDB | | |

The lecture claimed a column store reads about a fiftieth of the bytes. What
ratio did you measure, and why? (Count the columns of
`trips`.)

*Your answer:*

## Part B — Build the star schema (12 min)

Open `sql/star_schema.sql` (double-click it in the file browser on the left).
Read it top to bottom before you change anything.

**First, the grain.** On the `-- Grain:` line at the top, say in one sentence what
one row of `fact_trips` represents. Nothing in the file is written before this,
and the cell below refuses to build the schema until that line is filled in.

**Then `fact_trips`.** `dim_date` and `dim_zone` are written for you. Complete
`CREATE TABLE fact_trips` at the bottom of the file: the columns are listed there,
and the types, the primary key and the foreign keys are yours to choose.

The rows themselves come from `sql/load_fact.sql`, which is written for you.
Read it — it is where session 4's fan-out is dealt with, once.

In [6]:
def build_star() -> None:
    """Create the star from sql/star_schema.sql, then load it with sql/load_fact.sql."""
    schema = (SQL_DIR / "star_schema.sql").read_text()
    if not re.search(r"^-- Grain:[ \t]*\S", schema, re.MULTILINE):
        print("No grain yet. State it on the '-- Grain:' line of sql/star_schema.sql,")
        print("save the file, and run this cell again.")
        return

    run_file(SQL_DIR / "star_schema.sql")
    run_file(SQL_DIR / "load_fact.sql")

    print(sql("""
        SELECT (SELECT count(*) FROM dim_date)   AS dim_date,
               (SELECT count(*) FROM dim_zone)   AS dim_zone,
               (SELECT count(*) FROM fact_trips) AS fact_trips,
               (SELECT count(*) FROM trips)      AS trips_in_source
    """).to_string(index=False))
    # Every key on a fact row must find its dimension row, or that row
    # silently drops out of any query that joins to the dimension.
    print(sql("""
        SELECT count(*) FILTER (WHERE d.date_key IS NULL) AS dates_not_found,
               count(*) FILTER (WHERE z.zone_key IS NULL) AS zones_not_found
        FROM fact_trips f
        LEFT JOIN dim_date d ON d.date_key = f.date_key
        LEFT JOIN dim_zone z ON z.zone_key = f.zone_key
    """).to_string(index=False))


build_star()

No grain yet. State it on the '-- Grain:' line of sql/star_schema.sql,
save the file, and run this cell again.


### One request, two schemas

Request 3 from the lecture: **tip rate by payment type and borough**, here over the
whole of January. Against the normalised schema it is written for you.

In [ ]:
normalised = sql("""
SELECT p.method, z.borough,
       round(sum(t.tip) / sum(t.fare), 4) AS tip_rate
FROM trips t
JOIN payments p ON p.trip_id = t.trip_id
JOIN zones z    ON z.zone_id = t.pickup_zone_id
GROUP BY p.method, z.borough
ORDER BY p.method, z.borough
""")
normalised

Now write it against the star: `fact_trips` and `dim_zone`. Name the columns
`method`, `borough` and `tip_rate`, as above, so the two answers can be compared.

In [ ]:
star = sql("""

""")
star

The rows on which the two answers disagree:

In [ ]:
if star.empty:
    print("write the star query above first")
else:
    both = normalised.merge(star, on=["method", "borough"], suffixes=("_normalised", "_star"))
    display(both[both["tip_rate_normalised"] != both["tip_rate_star"]])

Compare the two, in two sentences. How do the two queries differ in what they
join, and why? Do the answers agree everywhere — and if not, which one is right?

*Your answer:*

## Part C — Partitioning (optional)

For anyone with time left. One day's average fare, read from the single Parquet
file, then from the copy split into one file per day. In each plan, find how
much of the data DuckDB actually opened.

In [ ]:
duckdb_plan(f"""
SELECT avg(fare), count(*)
FROM '{WHOLE}'
WHERE pickup_at >= '2024-01-15' AND pickup_at < '2024-01-16'
""")

In [ ]:
duckdb_plan(f"""
SELECT avg(fare), count(*)
FROM read_parquet('{BY_DAY}/*/*.parquet', hive_partitioning = true)
WHERE pickup_date = '2024-01-15'
""")

In the second plan, `Scanning Files` says how many of the 31 files were opened.

The single file was not read whole either. It is stored in blocks (row groups),
and each block records the smallest and largest value of each column. The cell
below prints those statistics for `pickup_at`: how many blocks could contain a
trip from 15 January? The file happens to be in pickup order; shuffle the rows,
and every block would cover the whole month.

*Your answer: how much of the data was not read, in each case?*

In [ ]:
DUCK.sql(f"""
    SELECT row_group_id AS block, stats_min AS earliest, stats_max AS latest,
           row_group_num_rows AS trips
    FROM parquet_metadata('{WHOLE}')
    WHERE path_in_schema = 'pickup_at'
""").df()